# L14a: Bandit and Weighted Majority Algorithm Portfolio Management Problems
In this lecture, we explore the idea of using multi-armed bandit algorithms to manage a portfolio of assets (equity and ETFs). The goal is to allocate resources among different investment options to maximize the investory benefit while minimizing risk.

> __Learning Objectives:__
> 
> By the end of this lecture, you should be able to:
> Three learning objectives go here.

Let's get started!
___

## Examples
Today, we will use the following examples to illustrate key concepts:

> [▶ Let's build a risk-aware ticker picker binary Bernoulli bandit](CHEME-5660-L14a-Example-RiskAware-BBBP-Ticker-Picker-Fall-2025.ipynb). In this example, we will build a binary Bernoulli bandit to help us pick stock tickers based on their historical performance. We'll scale the returns relative to a benchmark (e.g., S&P 500) and use a risk-adjusted return metric to inform our decisions. We will use the ε-greedy algorithm to balance exploration and exploitation as we learn which tickers yield the highest returns relative to an alternative benchmark, with and without risk adjustment.
___

<div>
    <center>
        <img src="figs/Fig-Bandits-Schematic.png" width="880"/>
    </center>
</div>

## Concept Review: Reinforcement Learning and Binary Bernoulli Bandits
The binary Bernoulli bandit problem is a special case of the stochastic bandit problem where the reward for taking action $a\in\mathcal{A}$ is binary $r_{t} = \left\{0,1\right\}$. The probability of getting reward `1` is unknown and needs to be estimated. The goal is to maximize the expected reward by selecting the best action at each time step.

* __Difference__: Unlike a completely general stochastic bandit problem, the binary Bernoulli bandit problem assumes the agent models how the world responds using a simple reward distribution, [the Bernoulli distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution). Thus, the agent has a model of the world.
* __Binary__: The reward distribution is binary. However, this is not as limiting as it may first appear. The experiment represented by the action $a$ can be a complex statement or function that evaluates to a boolean value. Thus, we can model many complex scenarios that evaluate to `true` or `false`.

The Bernoulli distribution is a discrete probability distribution that returns a value of `1` with probability $p$ and value `0` with probability $1-p$. The probability mass function of the Bernoulli distribution is given by:
$$
\begin{equation*}
\texttt{Bern}(r; p) = \begin{cases}
p & \text{if } r = 1,\\
1-p & \text{if } r = 0.
\end{cases}
\end{equation*}
$$
where $r\in\left\{0,1\right\}$ is the reward and $p\in[0,1]$ is the probability of getting reward `r = 1`. The expected reward of $X\sim\texttt{Bern}(r;p)$ is given by: $\mathbb{E}[X] = p$ and the variance is given by: $\text{Var}[X] = p(1-p)$.

The agent models the parameter $p$ using a probability distribution (e.g., [a Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution)) and updates this distribution as it observes rewards. This is the essence of the [Bayesian approach to bandit problems](https://onlinelibrary.wiley.com/doi/10.1002/asmb.874).

> __Why Beta distribution?__ The Beta distribution is the conjugate prior for the Bernoulli distribution. This means when we observe binary rewards (successes and failures), updating our belief about $p$ has a simple closed form: if we start with $\text{Beta}(\alpha, \beta)$ and observe $s$ successes and $f$ failures, the posterior is $\text{Beta}(\alpha + s, \beta + f)$. The parameters $\alpha$ and $\beta$ can be interpreted as prior successes and failures, making this approach both mathematically elegant and computationally efficient.

How do we solve this problem?

### $\epsilon$-Greedy Binary Bernoulli Bandit
The $\epsilon$-greedy algorithm is simple and effective for solving the binary Bernoulli bandit problem. The algorithm selects the _best action_ with probability $1-\epsilon$ and selects a random action with probability $\epsilon$. The pseudocode for the $\epsilon$-greedy algorithm is given below.

#### Pseudo-code
The agent has $K$ arms (choices), $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the total number of rounds is $T\gg{K}$. Initialize the parameters of [the Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) for each arm $a\in\mathcal{A}$ to $\alpha_{a} = 1$ and $\beta_{a} = 1$. The agent uses the following algorithm to choose which arm to pull (which action to take) during each round:

For $t = 1,2,\dots,T$:
1. _Initialize_: Roll a random number $p\in\left[0,1\right]$ and compute a threshold $\epsilon_{t}={t^{-1/3}}\cdot\left(K\cdot\log(t)\right)^{1/3}$.
2. _Exploration_: If $p\leq\epsilon_{t}$, choose a random (uniform) arm $a_{t}\in\mathcal{A}$. Execute the action $a_{t}$ and receive a stochastic reward $r_{t} \in \left\{0,1\right\}$.
3. _Exploitation_: Else if $p>\epsilon_{t}$, choose action $a^{\star}_{t}$, the action with the _highest expected probability of success_ (greedy choice), using the agent's model of the world. The highest probability action is: $a^{\star} = \arg\max_{a\in\mathcal{A}}\left\{\frac{\alpha(a) + \mathbf{S}(a)}{\alpha(a) + \beta(a) + \mathbf{S}(a) + \mathbf{F}(a)}\right\}$ where $\mathbf{S}(a)$ and $\mathbf{F}(a)$ are the number of successes and failures for arm $a$. Execute the action $a^{\star}_{t}$ and receive a stochastic reward $r^{\star}_{t}\in\left\{0,1\right\}$.
4. Update the success $\mathbf{S}(a^{\star})$ and failure $\mathbf{F}(a^{\star})$ arrays for the chosen arm $a^{\star}_{t}$ using the reward $r^{\star}_{t}$:
$$
\begin{equation*}
S(a^{\star}_{t}) \gets S(a^{\star}_{t}) + r^{\star}_{t},\quad F(a^{\star}_{t}) \gets F(a^{\star}_{t}) + (1-r^{\star}_{t})
\end{equation*}
$$

Using a model of the world allows the agent to make decisions about which actions to take. This is the essence of the Bayesian approach to bandit problems. The agent has a model of the likely reward distribution for _each_ action and uses this model to select the best action at each time step.

Let's look at an exampple of a risk-aware ticker picker binary Bernoulli bandit

> __Example__:
>
> 
> [▶ Let's build a risk-aware ticker picker binary Bernoulli bandit](CHEME-5660-L14a-Example-RiskAware-BBBP-Ticker-Picker-Fall-2025.ipynb). In this example, we will build a binary Bernoulli bandit to help us pick stock tickers based on their historical performance. We'll scale the returns relative to a benchmark (e.g., S&P 500) and use a risk-adjusted return metric to inform our decisions. We will use the ε-greedy algorithm to balance exploration and exploitation as we learn which tickers yield the highest returns relative to an alternative benchmark, with and without risk adjustment.

___

## Combinatorial Bandit Problems
In combinatorial bandit problems, the agent selects a subset of actions (a combination) at each time step. This is more complex than selecting a single action, as the number of possible combinations grows exponentially with the number of actions.

> __Difference with Standard Bandits?__
> 
> A combinatorial bandit problem extends the binary Bernoulli bandit framework to scenarios where decisions involve selecting __combinations of items__ rather than choosing a single arm. Each decision represents a configuration or subset of available options, making the action space combinatorial in nature.

In the __binary combinatorial bandit problem__, we have $K$ items, and each arm corresponds to a binary vector $\mathbf{a}\in\left\{0,1\right\}^{K}$ indicating which items are selected (1) or not selected (0). This leads to $N = 2^{K}$ possible arms to explore, where each arm represents a unique combination of the $K$ items.

For each round $t = 1,2,\dots,T$:
1. The agent selects an arm $\mathbf{a}_{t}\in\mathcal{A}$, where $\mathcal{A} = \left\{0,1\right\}^{K}$ is the set of all possible binary vectors of length $K$.
2. The agent receives a reward $r_{t}\in\mathbb{R}$ sampled from some (unknown) distribution associated with arm $\mathbf{a}_{t}$. This distribution is known by the world (nature) but unknown to the agent.
3. The agent updates its belief about the expected reward for the selected arm. $\texttt{GOTO}$ 1.

The goal is to maximize cumulative reward over $T$ rounds by learning which combinations of items yield the highest expected reward.

> __Key Challenges__
> 
> The combinatorial bandit problem introduces several challenges compared to the standard multi-armed bandit:
> * __Exponential growth__: With $K$ items, the number of possible arms grows as $N = 2^{K}$. Even modest values of $K$ lead to enormous action spaces. For example, with $K=10$ items, there are $N=1024$ arms to explore.
> * __Sparse exploration__: The agent may never visit most arms during the learning process. This makes learning difficult because many combinations remain unexplored.
> * __Structure exploitation__: Unlike the standard bandit problem, combinatorial bandits often exhibit structure. The reward for a combination may depend on interactions between items, not just individual item contributions. Exploiting this structure can improve learning efficiency.

The algorithms we developed for standard bandits (Explore-First, Epsilon-Greedy, UCB1, Thompson Sampling) can be adapted to the combinatorial setting. However, the exponential growth of the action space requires careful consideration of exploration strategies and computational efficiency.

### Combinatorial Epsilon-Greedy Algorithm
The combinatorial epsilon-greedy algorithm extends the standard epsilon-greedy approach to handle the exponential action space of combinatorial bandits. The key modification is that each arm is represented as a binary vector $\mathbf{a}\in\left\{0,1\right\}^{K}$, and the agent maintains average reward estimates for each of the $N = 2^{K}$ possible combinations.

#### Pseudo-code
The agent has $K$ items, leading to $N = 2^{K}$ possible arms (combinations), and the total number of rounds is $T$. Each arm $i\in\left\{1,2,\dots,N\right\}$ corresponds to a binary vector $\mathbf{a}\in\left\{0,1\right\}^{K}$.

_Initialization_: For each arm $i\in\left\{1,2,\dots,N\right\}$:
1. Generate the binary representation $\mathbf{a}_{i}$ using $i$ (e.g., $\mathbf{a}_{i} = \text{digits}(i, \text{base}=2, \text{pad}=K)$).
2. Execute action $\mathbf{a}_{i}$ and receive reward $r_{i}$ from the world.
3. Initialize the average reward estimate: $\mu_{i} \gets \mu_{0,i}\cdot\left(1-\frac{1}{T}\right) + \frac{1}{T}\cdot r_{i}$, where $\mu_{0,i}$ is an initial guess for arm $i$.

For rounds $t = 2,3,\dots,T$:
1. _Compute threshold_: Calculate $\epsilon_{t} = \frac{1}{t^{1/3}}\cdot\left(\log(K\cdot t)\right)^{1/3}$.
2. _Initialize_: Roll a random number $p\in\left[0,1\right]$.
3. _Exploration_: If $p\leq\epsilon_{t}$, randomly select an arm index $i\in\left\{1,2,\dots,N\right\}$ uniformly.
4. _Exploitation_: Else if $p>\epsilon_{t}$, choose the arm with the highest average reward: $i = \arg\max_{j\in\{1,\dots,N\}}\mu_{j}$.
5. _Generate action_: Convert arm index $i$ to binary vector $\mathbf{a}_{t} = \text{digits}(i, \text{base}=2, \text{pad}=K)$.
6. _Execute and observe_: Execute action $\mathbf{a}_{t}$ and receive reward $r_{t}$ from the world.
7. _Update estimate_: Update the average reward for arm $i$ using a weighted online average: $\mu_{i} \gets \mu_{i} + \frac{1}{t}\cdot\left(r_{t} - \mu_{i}\right)$.

__Output__: Return the history of rewards, final average reward estimates $\mu$, and action history.

> __Learning Rate Choice:__
>
> The learning rate $\alpha_t = \frac{1}{t}$ decreases over time. Each new observation receives less weight as more data is collected, while early observations retain their influence on the running average. This choice satisfies the Robbins-Monro conditions for stochastic approximation: $\sum_{t=1}^{\infty}\alpha_t = \infty$ and $\sum_{t=1}^{\infty}\alpha_t^2 < \infty$, which guarantee that the average converges to the true expected reward as the number of observations increases [1]. Alternative learning rates include:
> * **Sample mean**: $\alpha = \frac{1}{n_i}$ where $n_i$ is the number of times arm $i$ has been pulled. This gives equal weight to all observations of arm $i$ and converges to the true sample mean.
> * **Constant rate**: $\alpha = c$ for some fixed $c \in (0,1)$. This gives more weight to recent observations, allowing the algorithm to adapt to non-stationary environments.
> * **Polynomial decay**: $\alpha_t = \frac{1}{t^\beta}$ for $\beta \in (0,1]$. This balances between fast early learning and stable convergence.
>
> The choice of learning rate affects how quickly the estimates converge and whether they converge to the true expected reward. The $\frac{1}{t}$ schedule used here provides theoretical convergence guarantees while being simple to implement [1,2].

The combinatorial epsilon-greedy algorithm balances exploration and exploitation in the exponential action space by maintaining separate reward estimates for each combination. The weighted online average update allows the agent to adapt its estimates as it gathers more information, with the learning rate decreasing over time to stabilize the estimates.

Let's look at an example of a risk-aware portfolio manager using combinatorial bandits.

> __Example__
> 
> [▶ Let's build a risk-aware ticker picker binary Bernoulli bandit](CHEME-5660-L14a-Example-RiskAware-BBBP-Ticker-Picker-Fall-2025.ipynb). In this example, we will build a binary Bernoulli bandit to help us pick stock tickers based on their historical performance. We'll scale the returns relative to a benchmark (e.g., S&P 500) and use a risk-adjusted return metric to inform our decisions. We will use the ε-greedy algorithm to balance exploration and exploitation as we learn which tickers yield the highest returns relative to an alternative benchmark, with and without risk adjustment.

___

## Multiplicative Weights Algorithm (MWA)
The **Multiplicative Weights Algorithm (MWA)** is a simple yet robust online learning method that embodies a similar idea to the weighted majority algorithm, i.e., learning from expert advice. Here, the learning rate $\eta$ plays a role analogous to $\varepsilon$ in the Weighted Majority Algorithm, controlling adaptation speed. 

Let’s walk through the setup and sketch out the algorithm.

### Problem Setting
Suppose we are faced with a repeated decision-making task over rounds $t = 1, 2, \ldots, T$. At each round, we have access to $N$ experts, each providing a recommendation or prediction. Our goal is to combine their advice adaptively in order to make strong decisions over time, even in adversarial or uncertain environments.

* Let $\mathbf{p}^{(t)} = \{p_1^{(t)}, p_2^{(t)}, \ldots, p_N^{(t)}\}$ denote our belief distribution over experts at round $t$, updated iteratively based on their past performance.
* We select an expert by sampling from this distribution—for example, using a Categorical distribution:
  $i \sim \texttt{Categorical}(\mathbf{p}^{(t)})$—and follow that expert’s recommendation.
* After the decision is made, the environment (or adversary) reveals the true outcome. We then compute a cost vector $\mathbf{m}^{(t)} = \{m_1^{(t)}, \dots, m_N^{(t)}\}$, where $m_i^{(t)} \in [-1, 1]$ denotes the cost incurred by expert $i$ at time $t$. A correct prediction receives a cost of $-1$, and an incorrect one receives a cost of $+1$.

### Algorithm

__Initialize__: Fix a learning rate $\eta\leq{1}/{2}$, for each expert initialize the weight $w_{i}^{(1)} = 1$.

For $t=1,2,\dots,T$:
1. Chose expert $i$ with probability $p_{i}^{(t)} = w_{i}^{(t)}/\sum_{j=1}^{N}w_{j}^{(t)}$. Ask expert $i$ what the outcome of the experiment should be, denote the experts answer to this as: $\hat{y}_{i}^{(t)}$.
2. The adversary (nature) reveals the true outcome $y_{t}$ of the experiment at time $t$. Compute the cost of the following expert $i$, denoted as $m_{i}^{(t)}$. 
    $$
    m_i^{(t)} =
    \begin{cases}
    -1 & \text{if } \hat{y}_i^{(t)} = y_t \quad \text{(correct)} \\
    +1 & \text{if } \hat{y}_i^{(t)} \neq y_t \quad \text{(incorrect)}
    \end{cases}
   $$
3. Update the weights of expert $i$ as (renormalize the weights to obtain the new probability distribution):
$$
\begin{align*}
w_{i}^{(t+1)} = w_{i}^{(t)}\cdot\left(1-\eta\cdot{m_{i}^{(t)}}\right)
\end{align*}
$$

This is a super simple algorithm, with some very nice properties. The weights are updated multiplicatively based on the performance of each expert, hence the name Multiplicative Weights Algorithm. The learning rate $\eta$ controls how aggressively the algorithm adapts to the experts' performance. And there is a theoretical guarantee that the algorithm will perform nearly as well as the best fixed expert in hindsight, let's check that out!

### Theoretical Regret Bound
Assume all costs lie in the range $m_i^{(t)} \in [-1, 1]$, and fix a learning rate $\eta \leq \frac{1}{2}$. Then the Multiplicative Weights Algorithm (MWA) guarantees that for any expert $i$, after $T$ rounds:
$$
\begin{align*}
\sum_{t=1}^{T} \mathbf{p}^{(t)} \cdot \mathbf{m}^{(t)} & \leq \sum_{t=1}^{T} m_i^{(t)} + \eta \underbrace{\sum_{t=1}^{T} |m_i^{(t)}|}_{= T} + \frac{\ln N}{\eta} \\
\underbrace{\sum_{t=1}^{T} \mathbf{p}^{(t)} \cdot \mathbf{m}^{(t)} - \overbrace{\sum_{t=1}^{T} m_i^{(t)}}^{\text{best expert}}}_{R(T)} & \leq \eta T + \frac{\ln N}{\eta} \\
R(T) & \leq \eta T + \frac{\ln N}{\eta}\quad\blacksquare
\end{align*}
$$
where we used the fact that $|m_i^{(t)}| = 1$. By choosing $\eta = \sqrt{\frac{\ln N}{T}}$, this regret bound becomes sublinear:
$$
R(T) \leq 2 \sqrt{T \ln N}
$$
This ensures that the algorithm's **average regret per round** vanishes as $T \to \infty$, meaning that MWA performs nearly as well as the best fixed expert in hindsight.


Let's look at an example of a risk-aware portfolio manager using the multiplicative weights algorithm.

> __Example__
> 
> [▶ Let's build a risk-aware ticker picker binary Bernoulli bandit](CHEME-5660-L14a-Example-RiskAware-BBBP-Ticker-Picker-Fall-2025.ipynb). In this example, we will build a binary Bernoulli bandit to help us pick stock tickers based on their historical performance. We'll scale the returns relative to a benchmark (e.g., S&P 500) and use a risk-adjusted return metric to inform our decisions. We will use the ε-greedy algorithm to balance exploration and exploitation as we learn which tickers yield the highest returns relative to an alternative benchmark, with and without risk adjustment.

### Additional Resources
This module borrowed notes and was inspired from several sources: [Arora et al., The Multiplicative Weights Update Method: A Meta-Algorithm and Applications, Theory of Computing, Volume 8 (2012), pp. 121–164](https://theoryofcomputing.org/articles/v008a006/v008a006.pdf) and the [15-859 CMU Lecture 16](https://www.cs.cmu.edu/afs/cs.cmu.edu/academic/class/15859-f11/www/notes/lecture16.pdf) and [15-850 CMU Lecture 17](https://www.cs.cmu.edu/afs/cs.cmu.edu/academic/class/15859-f11/www/notes/lecture17.pdf). 
___

## Summary
One direct summary sentence of the lecture goes here.

> __Key Takeaways:__
> Three key takeaways go here.

One direct concluding sentence of the lecture goes here.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___